In [ ]:
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize
from itertools import product

# --- Pauli matrices ---
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
paulis = [I, X, Y, Z]

# --- Generate 15 Pauli tensor products (excluding I⊗I) ---
pauli_basis = []
for i, j in product(range(4), repeat=2):
    if (i, j) != (0, 0):
        pauli_basis.append(np.kron(paulis[i], paulis[j]))

# --- Target Hamiltonian H = X⊗X + Y⊗Y ---
H_target = np.kron(X, X) + np.kron(Y, Y)

# --- Q2 operator = (1 - Z⊗Z)/2 ---
Q2 = (np.eye(4) - np.kron(Z, Z)) / 2

# --- Generate U(alpha) = exp(-i H(alpha)) ---
def two_qubit_unitary_from_params(alpha):
    H = sum(a * b for a, b in zip(alpha, pauli_basis))
    H = (H + H.conj().T) / 2  # enforce Hermiticity
    U = expm(-1j * H)
    return U

# --- Loss function ---
def loss(alpha):
    U = two_qubit_unitary_from_params(alpha)

    # Diagonalization loss: minimize off-diagonal elements of U H U†
    H_eff = U @ H_target @ U.conj().T
    off_diag = H_eff - np.diag(np.diag(H_eff))
    loss_diag = np.linalg.norm(off_diag, 'fro')**2

    # Commutator loss: || [U, Q2] ||_F^2
    comm = U @ Q2 - Q2 @ U
    loss_comm = np.linalg.norm(comm, 'fro')**2

    # Total loss
    total_loss = loss_diag + loss_comm
    return total_loss


def loss(alpha):
    U = two_qubit_unitary_from_params(alpha)

    # Transformed Hamiltonian
    H_eff = U @ H_target @ U.conj().T

    # Diagonalization loss: off-diagonal Frobenius norm squared
    off_diag = H_eff - np.diag(np.diag(H_eff))
    loss_diag = np.linalg.norm(off_diag, 'fro')**2

    # Commutator loss: make [U, Q2] ≈ 0
    comm = U @ Q2 - Q2 @ U
    loss_comm = np.linalg.norm(comm, 'fro')**2

    # Total loss: diagonalize while commuting
    return loss_diag + loss_comm




In [ ]:
# Initial random parameters
np.random.seed(42)
alpha0 = np.random.randn(15)

# Optimize using BFGS
result = minimize(loss, alpha0, method='BFGS', options={'disp': True, 'maxiter': 500})
alpha_opt = result.x


In [ ]:
U_opt = two_qubit_unitary_from_params(alpha_opt)

# Check that U H U† is diagonal
H_eff = U_opt @ H_target @ U_opt.conj().T
print("Diagonalized H_target:\n", np.round(H_eff, 4))

# Check commutation with Q2
comm = U_opt @ Q2 - Q2 @ U_opt
print("\nCommutator norm ||[U, Q2]||_F =", np.linalg.norm(comm, 'fro'))

print("\nU:\n", np.round(U_opt, 5))


In [ ]:
from cirq import kak_decomposition

In [ ]:
kak_decomposition(U_opt, atol=1e-8)

In [ ]:
import cirq

In [ ]:
import numpy as np
from scipy.optimize import minimize

# Pauli matrices and identity
X = np.array([[0,1],[1,0]])
Y = np.array([[0,-1j],[1j,0]])
Z = np.array([[1,0],[0,-1]])
I = np.eye(2)

def kron(a,b):
    return np.kron(a,b)

# Operators
H = kron(X,X) + kron(Y,Y)
Q2 = (np.eye(4) - kron(Z,Z))/2

# Single-qubit unitary from Euler angles
def single_qubit_unitary(alpha, beta, gamma):
    Rz1 = np.array([[np.exp(-1j*alpha/2), 0],[0, np.exp(1j*alpha/2)]])
    Ry  = np.array([[np.cos(beta/2), -np.sin(beta/2)], [np.sin(beta/2), np.cos(beta/2)]])
    Rz2 = np.array([[np.exp(-1j*gamma/2), 0],[0, np.exp(1j*gamma/2)]])
    return Rz1 @ Ry @ Rz2

# Build U with real corners and single-qubit inner block
def build_U(params):
    theta = params[0]
    alpha, beta, gamma = params[1:4]
    R = np.array([[np.cos(theta), np.sin(theta)], [-np.sin(theta), np.cos(theta)]])  # real orthogonal
    V = single_qubit_unitary(alpha, beta, gamma)
    U = np.zeros((4,4), dtype=complex)
    U[0,0], U[0,3] = R[0,0], R[0,1]
    U[3,0], U[3,3] = R[1,0], R[1,1]
    U[1:3,1:3] = V
    return U

# Objective: minimize off-diagonal of U† H U + penalty for commutation with Q2
def objective(params, lambda_comm=10.0):
    U = build_U(params)
    H_eff = U.conj().T @ H @ U
    off_diag = H_eff - np.diag(np.diag(H_eff))
    commutator = U @ Q2 - Q2 @ U
    cost = np.linalg.norm(off_diag)**2 + lambda_comm * np.linalg.norm(commutator)**2
    return cost

# Initial guess
x0 = np.array([0.1, 0.1, 0.1, 0.1])

res = minimize(objective, x0, method='BFGS')

U_opt = build_U(res.x)
print("Optimized parameters:", res.x)
print("Cost:", objective(res.x))
print("Diagonalized H_eff:\n", np.round(U_opt.conj().T @ H @ U_opt, 6))
print("Commutator norm:", np.linalg.norm(U_opt @ Q2 - Q2 @ U_opt))


In [ ]:

# Check that U H U† is diagonal
H_eff = U_opt.conj().T @ H @ U_opt
print("Diagonalized H_target:\n", np.round(H_eff, 4))

# Check commutation with Q2
comm = U_opt @ Q2 - Q2 @ U_opt
print("\nCommutator norm ||[U, Q2]||_F =", np.linalg.norm(comm, 'fro'))

print("\nU:\n", np.round(U_opt, 5))

In [ ]:
from qibo.transpiler.unitary_decompositions import two_qubit_decomposition

In [ ]:
from qibo.backends import NumpyBackend
two_qubit_decomposition(0,1,U_opt,NumpyBackend())

In [ ]:
import sympy as sp
from sympy import sqrt, I

# Pauli matrices
X = sp.Matrix([[0, 1], [1, 0]])
Y = sp.Matrix([[0, -I], [I, 0]])
Z = sp.Matrix([[1, 0], [0, -1]])
I2 = sp.eye(2)

# Build 2-qubit operators
H = sp.kronecker_product(X, X) + sp.kronecker_product(Y, Y)
Z1Z2 = sp.kronecker_product(Z, Z)
Q2 = (sp.eye(4) - Z1Z2) / 2

# Define the analytic unitary: I ⊕ Hadamard in {|01>, |10>} subspace
s = 1/sqrt(2)
U = sp.Matrix([
    [1, 0, 0, 0],
    [0, s, s, 0],
    [0, s, -s, 0],
    [0, 0, 0, 1]
])

# Diagonalize H
H_eff = U.H * H * U
H_eff_diag = sp.diag(*H_eff.diagonal())
offdiag = H_eff - H_eff_diag

# Commutation check
commutator = U * Q2 - Q2 * U

# Print results
print("Diagonalized H (U† H U):")
sp.pprint(sp.simplify(H_eff))

print("\nOff-diagonal part (should be zero):")
sp.pprint(sp.simplify(offdiag))

print("\nCommutator [U, Q2] (should be zero):")
sp.pprint(sp.simplify(commutator))


In [ ]:
sp.pprint(U)

In [60]:
import numpy as np

# Pauli matrices
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

# Hamiltonian H = X⊗X + Y⊗Y
H = np.kron(X, X) + np.kron(Y, Y)

# Projector Q2 = (I - Z⊗Z)/2
Z1Z2 = np.kron(Z, Z)
Q2 = 0.5 * (np.eye(4) - Z1Z2)

# Construct U: identity on |00> and |11>, Hadamard on |01>, |10>
s = 1/np.sqrt(2)
U = np.array([
    [1, 0,   0,  0],
    [0, s,   -s,  0],
    [0, s,  s,  0],
    [0, 0,   0,  1]
], dtype=complex)

# Check if U diagonalizes H
H_eff = U @ H @ U.conj().T
Q2_eff = U @ Q2 @ U.conj().T
H_eff_diag = np.diag(np.diag(H_eff))
off_diag_norm = np.linalg.norm(H_eff - H_eff_diag)

# Check commutator with Q2
commutator = U @ Q2 - Q2 @ U
comm_norm = np.linalg.norm(commutator)

# Print results
print("U H U† =")
print(np.round(H_eff, 5))

print("\n U Q2 U† =")
print(np.round(Q2_eff, 5))

print("\nOff-diagonal norm of U†HU (should be ~0):", off_diag_norm)
print("Commutator norm [U, Q2] (should be ~0):", comm_norm)


U H U† =
[[ 0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j -2.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j -0.+0.j  2.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j]]

 U Q2 U† =
[[ 0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j -0.+0.j  0.+0.j]
 [ 0.+0.j -0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j]]

Off-diagonal norm of U†HU (should be ~0): 6.327514815575066e-17
Commutator norm [U, Q2] (should be ~0): 0.0


In [62]:
Q2 - Q2_eff

array([[0.00000000e+00+0.j, 0.00000000e+00+0.j, 0.00000000e+00+0.j,
        0.00000000e+00+0.j],
       [0.00000000e+00+0.j, 2.22044605e-16+0.j, 2.23711432e-17+0.j,
        0.00000000e+00+0.j],
       [0.00000000e+00+0.j, 2.23711432e-17+0.j, 2.22044605e-16+0.j,
        0.00000000e+00+0.j],
       [0.00000000e+00+0.j, 0.00000000e+00+0.j, 0.00000000e+00+0.j,
        0.00000000e+00+0.j]])

In [3]:
U

array([[ 1.        +0.j,  0.        +0.j,  0.        +0.j,
         0.        +0.j],
       [ 0.        +0.j,  0.70710678+0.j, -0.70710678+0.j,
         0.        +0.j],
       [ 0.        +0.j,  0.70710678+0.j,  0.70710678+0.j,
         0.        +0.j],
       [ 0.        +0.j,  0.        +0.j,  0.        +0.j,
         1.        +0.j]])

In [18]:
import numpy as np
nH = np.kron(H_eff, H_eff)
np.round(nH,5)

array([[ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j, -0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j, -0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.+0.j, -0.+0.j, -0.+0.j,
        -0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j, -0.+0.j,  0.+0.j,  0.+0.j, -0.+0.j,  4.-0.j, -0.+0.j,
        -0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.+0.j,
         0.+0

In [40]:
nH[10,10] # 5,5 6,6 9,9 10,10

np.complex128(3.9999999999999982+0j)

In [49]:
np.round(uu,2)

array([[ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j, -2.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j, -0.+0.j,  2.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -2.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -4.+0.j,  0.+0.j,
         0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,
         0.+0

In [43]:
uu = np.kron(H_eff,np.eye(4)) + np.kron(np.eye(4),H_eff)

In [57]:
X = np.array([[0, 1], [1, 0]])
uu = np.kron(Z, np.eye(2)) + np.kron(np.eye(2), Z)

In [58]:
np.round(uu,2)

array([[ 2.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j, -2.+0.j]])

In [4]:
from qibo import gates
gates.GIVENS(0,1,np.pi/4).matrix()

/home/alejandrosopena/envs/qibo_user1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[Qibo 0.2.17|INFO|2025-07-16 12:59:01]: Using qibojit (cupy) backend on /GPU:0


array([[ 1.        +0.j,  0.        +0.j,  0.        +0.j,
         0.        +0.j],
       [ 0.        +0.j,  0.70710678+0.j, -0.70710678+0.j,
         0.        +0.j],
       [ 0.        +0.j,  0.70710678+0.j,  0.70710678+0.j,
         0.        +0.j],
       [ 0.        +0.j,  0.        +0.j,  0.        +0.j,
         1.        +0.j]])

In [7]:
from qibo.symbols import Z
Z(0).matrix

array([[ 1.+0.j,  0.+0.j],
       [ 0.+0.j, -1.+0.j]])

In [ ]:
from qibo.symbols import Symbol

In [ ]:
symbols.np.array([0])

In [ ]:
2*Symbol(0,np.array([[1,0],[0,0]]))*Symbol(1,np.array([[0,0],[0,1]]))-2*Symbol(0,np.array([[0,0],[0,1]]))*Symbol(1,np.array([[1,0],[0,0]]))

In [ ]:
2*np.kron(np.array([[1,0],[0,0]]),np.array([[0,0],[0,1]]))-2*np.kron(np.array([[0,0],[0,1]]),np.array([[1,0],[0,0]]))

In [ ]:
op = 2*Symbol(0,np.array([[1,0],[0,0]]),'|0><0|')*Symbol(1,np.array([[0,0],[0,1]]),'|1><1|')-2*Symbol(0,np.array([[0,0],[0,1]]),'|1><1|')*Symbol(1,np.array([[1,0],[0,0]]),'|0><0|')

In [ ]:
from qibo.hamiltonians import SymbolicHamiltonian
from qibo.backends import NumpyBackend, construct_backend
from qibo.symbols import Symbol
import numpy as np

backend = construct_backend("qibojit")

zero_zero = backend.cast(np.array([[1,0],[0,0]]))
one_one = backend.cast(np.array([[0,0],[0,1]]))


op = 2*Symbol(0,zero_zero,'|0><0|')*Symbol(1,one_one,'|1><1|')-2*Symbol(0,one_one,'|1><1|')*Symbol(1,zero_zero,'|0><0|')



SymbolicHamiltonian(op, backend=backend).qu

[Qibo 0.2.17|WARNING|2025-07-15 14:27:39]: Calculating the dense form of a symbolic Hamiltonian. This operation is memory inefficient.


array([[ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  2.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j, -2.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j]])

In [ ]:
import qibo
from qibo.backends import set_backend
set_backend("qibojit")

In [ ]:
aa =Z(0)*Z(1)*Z(2)*Z(3)

In [ ]:
2*Symbol(0,zero_zero,'|0><0|').

In [6]:
from qiskit.circuit.library import RXXGate
import numpy as np
g1 = RXXGate(np.pi/4)

In [7]:
g1.to_matrix()

array([[ 0.92387953+0.j        ,  0.        +0.j        ,
         0.        +0.j        , -0.        -0.38268343j],
       [ 0.        +0.j        ,  0.92387953+0.j        ,
        -0.        -0.38268343j,  0.        +0.j        ],
       [ 0.        +0.j        , -0.        -0.38268343j,
         0.92387953+0.j        ,  0.        +0.j        ],
       [-0.        -0.38268343j,  0.        +0.j        ,
         0.        +0.j        ,  0.92387953+0.j        ]])

In [17]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator
import numpy as np

# Create the GIVENS matrix
s = 1/np.sqrt(2)
givens_matrix = np.array([
    [1, 0, 0, 0],
    [0, s, -s, 0],
    [0, s, s, 0],
    [0, 0, 0, 1]
], dtype=complex)

# Create circuit with UnitaryGate
circ = QuantumCircuit(2)
givens_gate = UnitaryGate(givens_matrix)
circ.append(givens_gate, [0, 1])

# Transpile to decompose into basic gates
transpiled_circ = transpile(circ, 
                           basis_gates=['rz', 'cx', 'sx'],  # Specify basis gates
                           optimization_level=3)     # No optimization to see full decomposition

print("Original circuit:")
print(circ.draw())
print("\nTranspiled circuit (decomposed):")
print(transpiled_circ.draw())

# Count gates
gate_counts = transpiled_circ.count_ops()
print(f"\nGate counts: {gate_counts}")

# Get the number of single and two-qubit gates
single_qubit = sum(count for gate, count in gate_counts.items() if gate in ['u', 'u1', 'u2', 'u3', 'rx', 'ry', 'rz', 'h', 's', 't'])
two_qubit = sum(count for gate, count in gate_counts.items() if gate in ['cx', 'cy', 'cz', 'cnot'])

print(f"Single-qubit gates: {single_qubit}")
print(f"Two-qubit gates: {two_qubit}")

# Verify the decomposition is correct
original_unitary = Operator(circ).data
decomposed_unitary = Operator(transpiled_circ).data
is_equivalent = np.allclose(original_unitary, decomposed_unitary, atol=1e-10)
print(f"\nDecomposition is correct: {is_equivalent}")

Original circuit:
     ┌──────────┐
q_0: ┤0         ├
     │  Unitary │
q_1: ┤1         ├
     └──────────┘

Transpiled circuit (decomposed):
     ┌───────────┐┌────┐┌──────────┐        ┌────┐  ┌──────────┐┌────┐     »
q_0: ┤ Rz(-5π/8) ├┤ √X ├┤ Rz(-π/2) ├──■─────┤ √X ├──┤ Rz(-π/4) ├┤ √X ├──■──»
     └┬──────────┤├────┤└──────────┘┌─┴─┐┌──┴────┴─┐└──────────┘└────┘┌─┴─┐»
q_1: ─┤ Rz(7π/8) ├┤ √X ├────────────┤ X ├┤ Rz(π/4) ├──────────────────┤ X ├»
      └──────────┘└────┘            └───┘└─────────┘                  └───┘»
«     ┌──────────┐    ┌────┐   ┌──────────┐
«q_0: ┤ Rz(-π/2) ├────┤ √X ├───┤ Rz(5π/8) ├
«     └──┬────┬──┘┌───┴────┴──┐└──────────┘
«q_1: ───┤ √X ├───┤ Rz(-7π/8) ├────────────
«        └────┘   └───────────┘            

Gate counts: OrderedDict([('rz', 8), ('sx', 6), ('cx', 2)])
Single-qubit gates: 8
Two-qubit gates: 2

Decomposition is correct: True
